# **ETL — Extract, Transform, Load**

## Objectives

* Extract the raw UCI Bank Marketing dataset and confirm it loads correctly
* Assess data quality: shape, data types, duplicates, and missing values
* Treat values recorded as "unknown" as missing data rather than as a valid category
* Recode pdays, where 999 indicates the client was never previously contacted
* Engineer features required for later analysis: age band, prior contact flag, and a binary target
* Apply data minimisation by removing any field not needed for the stated business requirements
* Save a cleaned, versioned dataset for use in the EDA, visualisation and modelling notebooks

## Inputs

* Data_Set/raw_data/bank-additional/bank-additional-full.csv — 41,188 records, 20 input variables, semicolon-delimited
* Data_Set/raw_data/bank-additional/bank-additional-names.txt — the data dictionary supplied with the dataset

## Outputs

* Data_Set/clean_data/v1/bank_marketing_cleaned.csv — the cleaned dataset used by all downstream notebooks
* Data_Set/outputs/v1/data_quality_summary.csv — a record of missingness and data quality issues found before cleaning

## Additional Comments

* Source: Moro, S., Rita, P., & Cortez, P. (2014). Bank Marketing [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306. Licensed under CC BY 4.0.

* Framing: the data relates to telephone marketing campaigns run by a Portuguese bank between May 2008 and November 2010. This project treats it as a proxy for a UK retail bank's outbound campaign, and assesses the resulting targeting model against UK law and FCA rules.

* "unknown" values are retained as explicit missing values rather than dropped. The pattern of missingness is itself a finding, and removing those records would silently exclude clients whose data was never captured.

* The "duration" field is retained at this stage but excluded from modelling. The dataset documentation states it is not known before a call takes place and should be discarded for a realistic predictive model. This is tested statistically in notebook 04 rather than accepted on documentation alone.

* The dataset contains no direct identifiers. Age, job, marital status and education are retained because they are required to assess whether the targeting model produces disproportionate outcomes; they are not used as model inputs in the final model.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop/fair-marketing-analytics/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop/fair-marketing-analytics'

# Section 1 — Load and Assess

Section 1 content

In [4]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

version = 'v1'
clean_dir = f'Data_Set/clean_data/{version}'
output_dir = f'Data_Set/outputs/{version}'

os.makedirs(clean_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print(f"Clean data: {clean_dir}")
print(f"Outputs:    {output_dir}")

Clean data: Data_Set/clean_data/v1
Outputs:    Data_Set/outputs/v1


In [5]:
raw_path = 'Data_Set/raw_data/bank-additional/bank-additional-full.csv'
df_raw = pd.read_csv(raw_path, sep=';')

print(f"Loaded {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns")
df_raw.head()

Loaded 41,188 rows and 21 columns


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## Data quality assessment

The dataset contains no null values in the conventional sense. Missing information is
recorded as the string "unknown", which pandas reads as a valid category rather than as
missing data. This assessment identifies where that occurs so it can be treated in
Section 2.

In [6]:
quality = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'nulls': df_raw.isnull().sum(),
    'unknowns': [(df_raw[c] == 'unknown').sum() if df_raw[c].dtype == 'object' else 0
                 for c in df_raw.columns],
    'unique_values': df_raw.nunique()
})
quality['unknown_pct'] = (quality['unknowns'] / len(df_raw) * 100).round(2)

quality.to_csv(f'{output_dir}/data_quality_summary.csv')

print(f"Duplicate rows: {df_raw.duplicated().sum()}")
quality.sort_values('unknown_pct', ascending=False)

Duplicate rows: 12


,dtype,nulls,unknowns,unique_values,unknown_pct
default,object,0,8597,3,20.87
education,object,0,1731,8,4.20
housing,object,0,990,3,2.40
loan,object,0,990,3,2.40
job,object,0,330,12,0.80
marital,object,0,80,4,0.19
age,int64,0,0,78,0.00
poutcome,object,0,0,3,0.00
nr.employed,float64,0,0,11,0.00
euribor3m,float64,0,0,316,0.00


In [7]:
both = ((df_raw['housing'] == 'unknown') & (df_raw['loan'] == 'unknown')).sum()
print(f"Records with 'unknown' for both housing and loan: {both} of 990\n")

print("default field:")
print(df_raw['default'].value_counts())

Records with 'unknown' for both housing and loan: 990 of 990

default field:
default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64


### Findings

- 41,188 records, 21 columns, 12 duplicate rows.
- The target is heavily imbalanced: 11.3% subscribed, 88.7% did not. A model predicting
  "no" for every client would be 88.7% accurate and entirely useless. Accuracy is
  therefore not a valid headline metric for this project.
- Six fields contain "unknown" values. The largest is `default` at 20.9%, meaning credit
  default status is unrecorded for a fifth of clients — the field most closely related
  to a client's financial position is the least reliably captured.
- `housing` and `loan` are each missing on exactly 990 records, indicating a common
  collection failure rather than random missingness.
- These gaps are retained as explicit missing values rather than dropped. Excluding them
  would silently remove clients whose data was never captured, and the pattern of
  missingness is itself relevant to the fairness assessment in Notebook 04.

---

# Section 2

Section 2 content

---

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.